# AI 我知道：Entropy 到底是什么？

## 1. 从“意外程度”到“平均信息量”

很多机器学习方法都会遇到 entropy。决策树用它衡量节点是否“纯”，Random Forest 中的单棵分类树也可以用它选择 split，分类模型会用到 cross-entropy loss，因果推断中的 entropy balancing 又把它用于样本加权。看起来这些场景差别很大，但它们都在处理同一个问题：**如何衡量不确定性、信息量或分布错配？**

如果把它放到我们熟悉的计量框架里，可以做一个谨慎的类比：OLS 中的 RSS 衡量残差平方和；MLE 中的 log-likelihood 衡量在给定参数下，样本观测值对模型参数的相对支持程度。对于离散模型，它可以写成样本联合概率的对数；对于连续模型，它对应联合密度的对数。因此，似然值本身不能简单说成概率。

Entropy 的位置类似：它不是某个单一模型的专属目标函数，而是一把更基础的尺子，用来刻画一个概率分布的平均信息量，也常被解释为该分布的平均不确定性。

::: {.callout-note}
### 一个必要的修正

不要把 entropy 说成“一个变量平均还剩多少不确定性”。“还剩”这个词更适合 conditional entropy，因为它强调“已知某些信息之后”的剩余不确定性。对于普通 entropy，更准确的说法是：$H(X)$ 刻画随机变量 $X$ 所服从的概率分布 $P(X)$ 的平均信息量或平均不确定性。
:::


## 2. 哪句话更有信息量？

先看几个生活中的判断：

- 广州 7 月下雨了。
- 广州 1 月下雪了。
- 一枚公平硬币抛出正面。
- 一枚 99% 概率为正面的硬币抛出反面。

这些句子都在告诉我们某个事件发生了，但它们带来的信息量不同。越是本来不太可能发生的事情，一旦发生，越能改变我们的认知。广州 7 月下雨并不稀奇，广州 1 月下雪却会让人惊讶；公平硬币抛出正面很普通，几乎总是正面的硬币突然抛出反面则更有信息量。

信息论把这个直觉写成一个非常简洁的公式。若事件 $x$ 发生的概率是 $p(x)$，则它的自信息量为：

$$
h(x)=-\log_2 p(x)
$$

这个定义有三个很自然的性质。第一，$p(x)$ 越小，$h(x)$ 越大。第二，如果 $p(x)=1$，则 $h(x)=0$，因为完全确定的事情不会带来新信息。第三，使用 $\log_2$ 时，单位是 bits，表示用二进制编码时平均需要多少信息单位。

![](./figs/entropy_intro_fig01_self_information.png){width="72%"}


In [1]:
# 这个代码块可以独立运行，用于计算几个事件的信息量。
import numpy as np

for p in [0.5, 0.1, 0.01, 1.0]:
    h = -np.log2(p)
    print(f"p = {p:>4}, h(x) = {h:.3f} bits")


p =  0.5, h(x) = 1.000 bits
p =  0.1, h(x) = 3.322 bits
p = 0.01, h(x) = 6.644 bits
p =  1.0, h(x) = -0.000 bits


## 3. 为什么要用 log？

使用 $-\log p(x)$ 不是任意选择。它至少满足两种重要直觉。

第一，概率越小，信息量越大。$p(x)=0.01$ 的事件比 $p(x)=0.5$ 的事件更令人意外，因此应有更高的信息量。

第二，独立事件的信息量可以相加。若 $A$ 和 $B$ 独立，则 $P(A,B)=P(A)P(B)$。使用 log 后有：

$$
-\log P(A,B)=-\log P(A)-\log P(B)
$$

这意味着两个独立事件一起发生的信息量，等于两个事件各自信息量之和。这个性质非常重要，因为它让信息量可以像“长度”“成本”一样累加。

::: {.callout-tip}
### 与 log-likelihood 的连接

在独立同分布样本中，likelihood 往往是许多密度或概率项的乘积。取 log 后，乘积变成加总，这就是 log-likelihood 在估计中如此方便的原因。Entropy 中的 log 也有类似作用：它让概率事件的信息量具备可加性。
:::


## 4. 从单个事件到整个分布

单个事件有信息量，但机器学习通常关心的是一个变量的全部可能取值。例如，一枚硬币有正面和反面；一个企业可能违约，也可能不违约；一个样本可能属于 A、B、C 三个类别之一。

若随机变量 $X$ 有若干可能取值 $x_1,x_2,\ldots,x_K$，其概率为 $p_1,p_2,\ldots,p_K$，则 entropy 定义为：

$$
H(X)=-\sum_{k=1}^{K}p_k\log_2 p_k
$$

这其实就是把每个事件的信息量 $-\log_2 p_k$ 按照其发生概率 $p_k$ 做加权平均：

$$
H(X)=\sum_{k=1}^{K}p_k h(x_k)
$$

因此，entropy 的直观含义是：**如果我们不断观察来自同一个分布的结果，那么每次观察平均带来多少信息量。**

![](./figs/entropy_intro_fig02_binary_entropy.png){width="70%"}


## 5. 三枚硬币的例子

考虑三枚硬币：

- 公平硬币：$P(H)=0.5, P(T)=0.5$。
- 偏硬币：$P(H)=0.75, P(T)=0.25$。
- 固定硬币：$P(H)=1, P(T)=0$。

公平硬币最难预测，因为正面和反面同样可能。固定硬币最容易预测，因为每次都是正面。偏硬币介于二者之间。对应的 entropy 分别为 1、约 0.811 和 0。

![](./figs/entropy_intro_fig03_coin_examples.png){width="74%"}


In [ ]:
import numpy as np

# 公平硬币
fair_h = -(0.5 * np.log2(0.5) + 0.5 * np.log2(0.5))
# 偏硬币
biased_h = -(0.75 * np.log2(0.75) + 0.25 * np.log2(0.25))
# 固定硬币
fixed_h = -(1.0 * np.log2(1.0))

print(f"公平硬币: H = {fair_h:.3f} bits")
print(f"偏硬币: H = {biased_h:.3f} bits")
print(f"固定硬币: H = {fixed_h:.3f} bits")

公平硬币: H = 1.000 bits
偏硬币: H = 0.811 bits
固定硬币: H = -0.000 bits


## 6. Entropy 衡量的不是“变量名”，而是“概率分布”

这一点很容易被忽略。我们不能说“天气这个变量的 entropy 一定高”或“违约这个变量的 entropy 一定低”。同一个变量，在不同样本、不同地区、不同时间窗口中，会有不同的概率分布，因此也会有不同的 entropy。

比如，

- 「是否下雨」是同一个变量，但广州 7 月和广州 1 月的降雨概率不同，entropy 也不同；
- 「企业是否违约」是同一个变量，但样本若只包含高风险企业，其违约率接近 50%，分类不确定性可能很高；若样本主要是稳健企业，违约率很低，entropy 反而较低。

::: {.callout-warning}
### 不要把 entropy 简单翻译成“混乱程度”

“混乱程度”可以帮助入门，但容易造成误解。更准确的说法是：entropy 是概率分布的函数，用来度量该分布下结果的平均信息量或平均不确定性。若所有结果接近等概率，则 entropy 高；若某些结果几乎必然发生，则 entropy 低。
:::


## 7. 机器学习为什么需要这把尺子？

机器学习模型经常要做三类判断。

第一，某个样本集合是否已经足够“纯”。例如，决策树在某个节点上，希望判断这个节点里的样本是否大多属于同一类。如果节点里全是同一类，entropy 为 0；如果多个类别混在一起，entropy 较高。

第二，某个特征是否能减少不确定性。例如，知道企业资产负债率之后，我们对其是否违约的不确定性是否下降？这会引出 conditional entropy 和 information gain。

第三，模型给出的预测概率是否贴近真实标签。例如，真实结果是违约，而模型只给了 1% 的违约概率，这会被 cross-entropy loss 给予很大惩罚。

所以，entropy 不是一个孤立概念。它是后面理解 decision tree、Random Forest、cross-entropy、KL divergence 和 entropy balancing 的共同入口。


## 8. 小结

这一篇只需要记住四句话。

- 单个事件的信息量是 $h(x)=-\log_2 p(x)$，事件越罕见，信息量越大。
- Entropy 是单个事件信息量的概率加权平均：$H(X)=-\sum_k p_k\log_2 p_k$。
- Entropy 刻画的是概率分布的平均信息量或平均不确定性，不是变量名称本身的性质。
- 当概率分布越均衡时，entropy 越高；当某个结果几乎确定时，entropy 越低。

下一篇会进一步讨论：如果我们已经知道一些信息，例如月份、气温区间、企业规模或财务指标，那么原来的不确定性会减少多少？这就进入 conditional entropy 和 information gain。


## 参考文献

- Shannon, C. E. (1948). A mathematical theory of communication. *Bell System Technical Journal*, 27(3), 379-423. [Link](https://doi.org/10.1002/j.1538-7305.1948.tb01338.x), [PDF](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf), [Google](https://scholar.google.com/scholar?q=A+Mathematical+Theory+of+Communication+Shannon).
- Shannon, C. E. (1948). A mathematical theory of communication. *Bell System Technical Journal*, 27(4), 623-656. [Link](https://doi.org/10.1002/j.1538-7305.1948.tb00917.x), [PDF](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf), [Google](https://scholar.google.com/scholar?q=A+Mathematical+Theory+of+Communication+Shannon).
